# QICK Lab-Readout Driver

Thin operator-in-the-loop driver over the `labreadout` package. Each step **runs → fits → plots → suggests** the next scan; nothing is committed to `calibration.yml` until you call `.accept()`.

- Hardware setup lives in `hardware.yml` (edit by hand).
- Calibrated values persist in `calibration.yml` (managed by `.accept()`).
- Raw measurement CSVs are written by `quick` in the **unchanged** format; fit results are saved to `*.fit.yml` sidecars alongside them.

## Notebook Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import labreadout as lr

# Connects to the QICK board, applies RF/bias setup from hardware.yml,
# and merges calibration.yml over the var defaults.
sess = lr.Session.connect('hardware.yml', 'calibration.yml')
sess.var

## 1. Board & Resonator Bringup
Loopback first to confirm the readout chain, then a wide resonator scan.

In [ ]:
# Hardware can't be rewired -- so verify the logical->physical port map rather
# than guess it. connect() already printed it; re-check it any time:
sess.check_ports()

# Loopback bring-up: start at r_offset=0, measure the readout-window delay from
# where the pulse lands, then sweep power and pick a level above noise but below
# ADC over-range. The advisory report prints automatically; review, then accept.
lb = sess.loopback_calibrate(powers=range(-40, 1, 5))  # fullscale from hardware.yml expected:
lb.plot(); plt.show()
print(lb.suggestion)
lb.accept()            # commits r_offset + r_power (or override the recs and re-run)

In [ ]:
# Coarse resonator spectroscopy. Continuous pulse + long readout window.
res = sess.resonator_spectroscopy(
    r_freq=np.arange(6500, 6750, 0.1),
    title='resonator_coarse',
    r_power=-20, p0_mode='periodic', r1_length=213, hard_avg=100,
)
res.plot(); plt.show()
print(res.suggestion)
# An advisory health report (SNR / over-range / known-value sanity) prints above.
# Recommended next-step knobs live in res.recommendations:
print(res.diagnosis.status, '->', res.recommendations)

In [ ]:
# Fine scan around the suggested window, then commit the resonator frequency.
lo, hi = lr.adaptive.next_window(res.fit.f0, res.fit.hwhm, span_factor=8, min_span=4)
res = sess.resonator_spectroscopy(
    r_freq=np.arange(lo, hi, 0.01),
    title='resonator_fine', r_power=-20, p0_mode='periodic', r1_length=213,
)
res.plot(); plt.show()
print(res.suggestion)
res.accept()   # writes r_freq -> calibration.yml

## 2. Dispersive / Power Sweep
Track the resonance vs. readout power to find the low-power (dispersive) shift.

In [ ]:
for power in [-30, -25, -20, -15, -10]:
    r = sess.resonator_spectroscopy(
        r_freq=np.arange(res.fit.f0 - 3, res.fit.f0 + 3, 0.02),
        title=f'resonator_power_{power}', r_power=power, p0_mode='periodic',
    )
    print(power, 'dB ->', round(r.fit.f0, 4), 'MHz')

## 3. IQ Readout Calibration
Single-shot ground/excited clouds → separation axis, threshold, fidelity.
Run an `IQScatter` for |g> and |e>, then fit the threshold.

In [ ]:
# g, e = sess._run('IQScatter', 'iq_g', {}, q_gain=0.0), sess._run('IQScatter', 'iq_e', {}, q_gain=v_pi)
# Ig, Qg = g[0][:, 3], g[0][:, 4]; Ie, Qe = e[0][:, 3], e[0][:, 4]
# iq = lr.fitting.fit_iq_threshold(Ig, Qg, Ie, Qe)
# iq.plot(); plt.show(); print(iq.suggestion)
# sess.var['r_threshold'] = iq.threshold; lr.state.record('calibration.yml', {'r_threshold': iq.threshold})

## 4. Qubit Spectroscopy
Coarse scan to locate the qubit, then a fine scan around the suggested window.

In [ ]:
qub = sess.qubit_spectroscopy(
    q_freq=np.arange(4800, 5400, 2.0),
    title='qubit_spec_coarse', q_gain=0.2, hard_avg=1000,
)
qub.plot(); plt.show()
print(qub.suggestion)
# If the report above says noise-dominated, raise hard_avg / q_gain before the
# fine scan instead of re-running blindly (this was the 2026-06-29 time sink):
print(qub.diagnosis.status, '->', qub.recommendations)

In [ ]:
lo, hi = lr.adaptive.next_window(qub.fit.f0, qub.fit.linewidth, span_factor=8, min_span=10)
qub = sess.qubit_spectroscopy(
    q_freq=np.arange(lo, hi, 0.2), title='qubit_spec_fine', q_gain=0.2,
)
qub.plot(); plt.show()
print(qub.suggestion)
qub.accept()   # writes q_freq -> calibration.yml

## 5. Rabi Calibration
Sweep drive (gain or length) → decaying cosine → pi-pulse value.

In [ ]:
rabi = sess.rabi(
    x=np.arange(0, 0.5, 0.005), xlabel='Pulse length (us)',
    title='rabi_length', q_length=np.arange(0, 0.5, 0.005),
)
rabi.plot(); plt.show()
print(rabi.suggestion)
# rabi.recommendations -> {'q_length': pi_value}. Stage it for later steps with
# sess.apply(...), and commit the pi value to the knob you drive:
sess.apply(rabi.recommendations)
rabi.accept(extra=rabi.recommendations)

## 6. T1 Relaxation

In [ ]:
t1 = sess.t1(
    time=np.arange(0, 10, 0.05), title='T1', rep=5, hard_avg=20000,
)
t1.plot(); plt.show()
print(t1.suggestion)
t1.accept(extra={'r_relax': 5 * t1.fit.t1})

## 7. T2 Ramsey / Echo

In [ ]:
t2 = sess.t2(
    time=np.arange(0, 10, 0.05), experiment='T2Ramsey', title='T2_ramsey',
)
t2.plot(); plt.show()
print(t2.suggestion)
# Correct the qubit frequency by the measured detuning if desired:
# sess.var['q_freq'] += t2.fit.detuning; lr.state.record('calibration.yml', {'q_freq': sess.var['q_freq']})

---
### Notes
- `sess.var` is the live working copy; `.accept()` is the only thing that writes `calibration.yml`.
- Per-call keyword overrides (e.g. `r_power=-20`) win over both the config defaults and the calibration state for that single run.
- Every fitted step also drops a `*.fit.yml` sidecar next to its raw CSV.